In [1]:
import os
import sys
import random
from itertools import chain
from utils.data_utils import read_titles, generate_examples, generate_examples_from_lines, get_spans_bio
from argparse import ArgumentParser
# from torchblocks.utils.seed import seed_everything 


# 通过标注和未标注数据生成标题集和和实体集和

In [22]:
parser = ArgumentParser()
parser.add_argument("--version", type=str, default="pretrain-v0")
parser.add_argument("--output_dir", type=str, default="data/processed/")
parser.add_argument("--min_length", type=int, default=0)
parser.add_argument("--max_length", type=int, default=256)
parser.add_argument("--train_ratio", type=float, default=0.8)
parser.add_argument("--seed", type=int, default=42)
# args = parser.parse_args()
args, _ = parser.parse_known_args()

# seed_everything(args.seed)
args.output_dir = os.path.join(args.output_dir, args.version)    
os.makedirs(args.output_dir, exist_ok=True)

In [17]:
corpus = []
entities = set()
for example in chain(
    generate_examples("data/中文NER数据集/商品标题2022-NER/train.txt"),
    generate_examples("data/中文NER数据集/商品标题2022-NER/preliminary_test_a/word_per_line_preliminary_A.txt"),
    generate_examples("data/中文NER数据集/商品标题2022-NER/preliminary_test_b/word_per_line_preliminary_B.txt"),
    generate_examples_from_lines("data/中文NER数据集/商品标题2022-NER/unlabeled_train_data.txt"),
    generate_examples("data/中文NER数据集/ecommerce/dev.txt"),
    generate_examples("data/中文NER数据集/ecommerce/train.txt"),
    generate_examples("data/中文NER数据集/ecommerce/test.txt")
):
    text = "".join(example[1]["tokens"])
    # 保存实体词典
    ner_tags = example[1]["ner_tags"] if "ner_tags" in example[1] else None
    if ner_tags is not None:
        for label, start, end in get_spans_bio(ner_tags):
            entities.add(text[start: end + 1])
        # import jieba
        # a = jieba.lcut(text)
        # jieba.load_userdict("data/processed/pretrain-v0/entities.txt")
        # b = jieba.lcut(text)
        # print()
    length = len(text)
    if length < args.min_length or length > args.max_length:
        continue
    corpus.append(text)
print(f"Corpus size: {len(corpus)}")


Corpus size: 1067996


In [19]:
with open(os.path.join(args.output_dir, "entities.txt"), "w", encoding="utf-8") as f:
    f.writelines([entity + "\n" for entity in entities])

In [23]:
random.seed(args.seed)
random.shuffle(corpus)
corpus = list(map(lambda x: x + "\n", corpus))
with open(os.path.join(args.output_dir, "corpus.txt"), "w", encoding="utf-8") as f:
    f.writelines(corpus)

In [24]:
if args.train_ratio is not None:
    num_corpus_train = int(len(corpus) * args.train_ratio)
    corpus_train = corpus[: num_corpus_train]
    corpus_valid = corpus[num_corpus_train: ]
    with open(os.path.join(args.output_dir, "corpus_train.txt"), "w", encoding="utf-8") as f:
        f.writelines(corpus_train)
    with open(os.path.join(args.output_dir, "corpus_valid.txt"), "w", encoding="utf-8") as f:
        f.writelines(corpus_valid)

# 初步清洗-考虑需不需要
- 删除不匹配的"()\[]"
- 删除没有意义特殊符号
- 删除html格式文本
- 删除隐藏字符2
- 先不洗

In [ ]:
import re

HTML_ENTITY_RE = re.compile(r'&[a-zA-Z]+;?')
MULTI_SPACE_RE = re.compile(r'\s+')

PAIR_SYMBOLS = [
    ('(', ')'),
    ('[', ']'),
    ('【', '】'),
    ('{', '}'),
]

def remove_invalid_pairs(text: str) -> str:
    for left, right in PAIR_SYMBOLS:
        if text.count(left) != text.count(right):
            # 数量不匹配 → 删除这一对符号
            text = text.replace(left, '').replace(right, '')
    return text


def clean_title(title: str) -> str:
    if not title:
        return ""

    # 1️⃣ 处理 HTML 实体
    title = HTML_ENTITY_RE.sub(' ', title)

    # 2️⃣ 清理乱码字符
    title = ''.join(
        ch if (
            ch.isalnum() or
            '\u4e00' <= ch <= '\u9fff' or
            ch in ' .,-_/()【】[]{}+*#%℃'
        ) else '^'
        for ch in title
    )

    # ⭐ 3️⃣ 删除不成对的双边符号
    title = remove_invalid_pairs(title)

    # 4️⃣ 合并多空格
    title = MULTI_SPACE_RE.sub(' ', title)

    # 5️⃣ 去首尾空格
    title = title.strip()

    return title

In [ ]:
# from tqdm import tqdm
# read_titles_path = os.path.join(args.output_dir, "..", "raw_titles.txt")
# titles = read_titles(read_titles_path)
# titles = [clean_title(title) for title in tqdm(titles, desc="Cleaning titles")]

Cleaning titles: 100%|██████████| 1067998/1067998 [00:04<00:00, 217623.55it/s]


# 根据整理的语料进行分词

In [ ]:
file_name = 'data/processed/pretrain-v0/corpus_train.txt'
with open(file_name, 'r', encoding='utf-8') as f:
    titles = f.readlines()
titles = [line.strip() for line in titles if len(line) > 0 and not line.isspace()]

In [33]:
from ltp import LTP
ltp_tokenizer = LTP().get_tokenizer()

KeyboardInterrupt: 

In [ ]:
bert_tokenizer = BertTokenizerZh.from_pretrained(args.bert, do_ref_tokenize=False)